# 02 - Total p-Variation (TpV) Reconstruction for Sparse-View CT

This notebook implements a model-based variational reconstruction method for sparse-view CT. The measured data are the degraded Mayo sinograms generated in notebook 01, and the reconstruction is obtained by solving the unconstrained optimization problem:

$$
\hat{x} = \arg\min_x \frac{1}{2}\|Kx-y^\delta\|_2^2 + \lambda \mathcal{R}_{TpV}(x)
$$

where:
- $K$ is the parallel-beam CT projector (Radon transform).
- $y^\delta$ is the noisy sinogram.
- $\mathcal{R}_{TpV}(x) = \sum_i \|\nabla x_i\|_2^p$ is the Total $p$-Variation prior, with $p$ in the non-convex sparse range $0.1 < p < 0.5$ as requested by the project specifications.

The solver is based on the **Chambolle-Pock Primal-Dual algorithm** (using `IPPy` tools) initialized from scratch (zeros) without any FBP initialization, to focus strictly on the variational paradigm. 

To comply with the project specifications (*"Students must ensure all methods use the same degraded inputs"*), **both this notebook and the ResUNet notebook evaluate on the exact same central test slice**, ensuring a perfectly fair and mathematically rigorous comparison between the variational and deep learning models.

## 1. Environment Setup and Imports

Mount the Google Drive, install `astra-toolbox` if needed, and import libraries from standard Python, PyTorch, and `IPPy`.

In [ ]:
!pip install astra-toolbox

from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path
import json
import numpy as np
import torch
import matplotlib.pyplot as plt

# Paths setup
PROJECT_ROOT = Path("/content/drive/MyDrive/LM_INFORMATICA/COMPUTATIONAL_IMAGING")
PROCESSED_DIR = PROJECT_ROOT / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tpv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(PROJECT_ROOT))

from IPPy import operators, solvers, utilities
from IPPy.utilities import normalize
from IPPy.utilities.metrics import PSNR, SSIM

# Set seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device configuration (CPU or GPU)
device = utilities.get_device()
print("Device used:", device)
print("Processed data directory:", PROCESSED_DIR)
print("TpV outputs directory:", OUTPUT_DIR)

## 2. Load the Preprocessed Data Contract (Test Patient)

Load the first test patient file to perform reconstruction on the exact same test slice used by downstream notebooks, ensuring consistency.

In [ ]:
manifest_path = PROCESSED_DIR / "manifest.json"
with manifest_path.open("r", encoding="utf-8") as file:
    manifest = json.load(file)

# Load the first test patient to guarantee identical evaluation inputs across all methods
test_split = manifest["splits"]["test"]
patient_record = test_split["patients"][0]
patient_path = PROCESSED_DIR / patient_record["path"]
payload = torch.load(patient_path, map_location="cpu")

clean_images = payload["clean"].to(torch.float32)
sinograms = {key: val.to(torch.float32) for key, val in payload["sinograms"].items()}
metadata = payload["metadata"]
patient_id = metadata["patient_id"]

print(f"Loaded Test Patient ID: {patient_id}")
print(f"Clean images batch shape: {tuple(clean_images.shape)}")
for views_key, sino in sinograms.items():
    print(f"Sinogram ({views_key} views) shape: {tuple(sino.shape)}")

## 3. Configure CT Projector and TpV Solver

Define the number of views to test (select from 180, 90, 60, 45) and initialize the parallel-beam CT projector $K$ and the unconstrained Chambolle-Pock TpV solver.

In [ ]:
# Set the angle setting to test (180, 90, 60, 45)
n_views = 60
angle_key = str(n_views)

if angle_key not in sinograms:
    raise ValueError(f"Angle config '{angle_key}' not found in preprocessed dataset.")

# Initialize physical parallel-beam projector
K = operators.CTProjector(
    img_shape=(256, 256),
    angles=np.linspace(0.0, np.pi, n_views),
    det_size=256,
    geometry="parallel",
    force_cpu=True,  # Set to False if CUDA/GPU support is fully configured in ASTRA
)

# Initialize the Chambolle-Pock TpV Solver
solver = solvers.ChambollePockTpVUnconstrained(K)
print(f"Initialized CT Projector and TpV Solver for {n_views} views.")

## 4. Hyperparameter Optimization on the Shared Test Slice

Run this cell to select the shared test slice, apply the Chambolle-Pock solver with your chosen $\lambda$ and $p$ hyperparameters, and compute the PSNR and SSIM. 

You can manually change the hyperparameters and rerun this cell multiple times to evaluate the performance on the same shared test slice.

In [ ]:
# -------------------- HYPERPARAMETERS TO TUNE --------------------
lmbda = 0.01          # Regularization parameter (weight of TpV penalty)
p = 0.35              # Sparsity parameter (strictly between 0.1 and 0.5)
maxiter = 150         # Number of maximum iterations
# -----------------------------------------------------------------

if not (0.1 < p < 0.5):
    raise ValueError(f"Project specifications require 0.1 < p < 0.5. Got p = {p}")

# Select the fixed central slice to guarantee identical evaluation inputs across all methods
slice_idx = clean_images.shape[0] // 2
print(f"Selected shared test slice index: {slice_idx}")

# Prepare 2D Ground Truth image and 2D corrupted sinogram on the selected device
x_true = clean_images[slice_idx : slice_idx + 1].to(device)
y_delta = sinograms[angle_key][slice_idx : slice_idx + 1].to(device)

print(f"Running Chambolle-Pock TpV solver (starting from scratch)... ")

# Execute the variational solver (starting_point=None starts from zeros)
x_sol, _ = solver(
    y_delta,
    lmbda=lmbda,
    starting_point=None,  # Standard zero-image initialization
    x_true=x_true,
    maxiter=maxiter,
    tolf=1e-5,
    tolx=1e-5,
    p=p,
    verbose=True,
)

# Normalize and clamp the final reconstructed image
x_sol = x_sol.detach().cpu()
x_true_cpu = x_true.detach().cpu()

x_sol_norm = normalize(x_sol).clamp(0.0, 1.0)

# Compute quantitative metrics according to project specifications (PSNR, SSIM)
psnr_val = PSNR(x_sol_norm, x_true_cpu)
ssim_val = SSIM(x_sol_norm, x_true_cpu)

print("\n--- QUANTITATIVE RESULTS ---")
print(f"PSNR: {psnr_val:.4f} dB")
print(f"SSIM: {ssim_val:.4f}")

## 5. Visual Inspection and Error Map

Plot the Ground Truth image, the TpV reconstruction, and the absolute error map $|TpV - GT|$ to inspect the spatial distribution of reconstruction errors. The generated visual comparison is saved in the `/outputs/tpv/` directory.

In [ ]:
# Convert to 2D numpy arrays for visualization
gt_np = x_true_cpu[0, 0].numpy()
recon_np = x_sol_norm[0, 0].numpy()
error_map = np.abs(recon_np - gt_np)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
fig.suptitle(f"TpV Reconstruction Panel - {n_views} views - Slice {slice_idx}\n(lmbda={lmbda}, p={p})", fontsize=14)

axes[0].imshow(gt_np, cmap="gray", vmin=0.0, vmax=1.0)
axes[0].set_title("Ground Truth")
axes[0].axis("off")

axes[1].imshow(recon_np, cmap="gray", vmin=0.0, vmax=1.0)
axes[1].set_title(f"TpV Reconstruction\nPSNR: {psnr_val:.2f} dB | SSIM: {ssim_val:.4f}")
axes[1].axis("off")

im_err = axes[2].imshow(error_map, cmap="magma")
axes[2].set_title("Absolute Error Map |TpV - GT|")
axes[2].axis("off")
fig.colorbar(im_err, ax=axes[2], shrink=0.8)

output_fig_path = OUTPUT_DIR / f"tpv_reconstruction_panel_{n_views}.png"
fig.savefig(output_fig_path, dpi=150)
plt.show()
plt.close(fig)

print("Saved TpV reconstruction panel:", output_fig_path)

## 6. Quantitative Results

Display the required PSNR and SSIM metrics for the TpV reconstruction.

In [ ]:
print("="*45)
print(f"{ 'METHOD':<20 } | { 'PSNR (dB)':<12 } | { 'SSIM':<8 }")
print("="*45)
print(f"{ 'TpV':<20 } | { psnr_val:<12.4f } | { ssim_val:<8.4f }")
print("="*45)